# TL;DR
This dataset records the sales of all product family of all stores in Ecuador 

- train.csv (2013-01-01 to 2017-08-15)
> Note: sales can be fractional (e.g., kilograms). > onpromotion is a count of items in that family under promotion.
- test.csv (2017-08-16 to 2017-08-31)
> Goal: Predict `sales` for these 16 days. Same features as train (minus the target).
- stores.csv: There is only 54 stores (store_nbr: 1-54) in these datasets, including both train and test dateset. 
- oil.csv: Daily oil price (critical for Ecuador's economy).
> Constraint: No weekend data (trading is closed).
> Constraint: Nulls occur on market holidays.
- holidays_events.csv: Metadata for National(in the entire Ecuador), Regional(in selected states), and Local(in selected cities) holidays.
> Note: Multiple holidays can overlap on a single day across different locales.  
- transactions.csv: Actual count of transactions per store/date.
> Conflict: Reveals that many stores were not active at the start of 2013, despite having "0 sales" rows in the train.csv.

### Assumption:  
Based on the misalignments in the transactions.csv and train.csv, we are applying the following logic:
- Staggered Starts: Stores opened at different times during the 2013–2017 window.
- Product Introductions: Not all product families (e.g., "BOOKS", "BABY CARE") began selling on the store's opening day.

> Action Suggested: Remove all "leading zeros"—any instances occurring before the first actual sale for a specific (Store + Family) pair to prevent the model from learning a false pattern of zeros for stores or products that didn't exist yet.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import os
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

In [ ]:
torch.cuda.is_available()
torch.__version__

<center><h2>Basic data preprocessing</h2></center>

In [ ]:
train=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/train.csv",parse_dates=["date"])
test=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/test.csv",parse_dates=["date"])
sub=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv")
oil=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/oil.csv",parse_dates=["date"])
stores=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/stores.csv")
holidays=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv",parse_dates=["date"])
txns=pd.read_csv(r"/kaggle/input/store-sales-time-series-forecasting/transactions.csv",parse_dates=["date"])

In [ ]:
stores.rename(columns={'type': 'store_type'}, inplace=True)

In [ ]:
holidays.rename(columns={'type': 'holiday_type'}, inplace=True)

In [ ]:
all_df = {
    'train': train,
    'test': test,
    'oil': oil,
    'stores': stores,
    'holidays': holidays,
    'txns': txns
}

for name, df in all_df.items():
    print(f"{name} shape:",df.shape)
    display(df.head())


In [ ]:
# 查看各个数据的基本信息
for name, df in all_df.items():
    print(f"{name} info:")
    df.info()
    print("\n\n")

In [ ]:
id = 'id'
timestamp = ['date']
target = ['sales']

CATEGORICAL = [
    ### train & test
    'store_nbr',
    'family',
    ### store
    'city',
    'state',
    'store_type',
    'cluster', 
    ### holiday
    'holiday_type',
    'locale',
    'locale_name',
    # 'description',
]

BOOLEAN = [
    ### train & test
    'onpromotion',
    ### holiday
    'transferred',
    
]

NUMERICAL = [
    ### oil
    'dcoilwtico',
    ### txn
    'transactions'
]

FEATURES = CATEGORICAL + BOOLEAN + NUMERICAL

In [ ]:
for name, df in all_df.items():
    columns = df.columns
    for col in columns:
        if col in CATEGORICAL:
            df[col] = df[col].astype('category')
        elif col in BOOLEAN:
            df[col] = df[col].astype("int8")
        elif col in NUMERICAL:
            df[col] = df[col].astype("float32")
            
    print(f"{name} info:")
    df.info()
    print("\n\n")

<center><h2>Functions Used for EDA</h2></center>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def univariate_analysis(df: pd.DataFrame, visualize: bool = False):
    """
    Perform univariate EDA with support for numerical, categorical, boolean,
    and datetime features. Optional visualization.
    """

    def evaluate_numerical_feature(series: pd.Series, col_name: str):
        desc = series.describe()

        stats = {
            "dtype": series.dtype,
            "count": desc["count"],
            "null_count": series.isna().sum(),
            "mean": desc["mean"],
            "std": desc["std"],
            "min": desc["min"],
            "25%": desc["25%"],
            "50% (median)": desc["50%"],
            "75%": desc["75%"],
            "max": desc["max"]
        }

        if visualize:
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            series.plot(kind="hist", bins=30, ax=axes[0], title=f"{col_name} - Histogram")
            series.plot(kind="box", ax=axes[1], title=f"{col_name} - Boxplot")
            plt.tight_layout()
            plt.show()

        return stats

    def evaluate_categorical_feature(series: pd.Series, col_name: str):
        stats = {
            "dtype": series.dtype,
            "count": series.count(),
            "null_count": series.isna().sum(),
            "n_unique": series.nunique(dropna=True),
            "value_distribution": series.value_counts(dropna=False, normalize=True).to_dict()
        }

        if visualize:
            series.value_counts(dropna=False).plot(
                kind="bar", figsize=(6, 4), title=f"{col_name} - Distribution"
            )
            plt.ylabel("Count")
            plt.show()

        return stats

    def evaluate_datetime_feature(series: pd.Series, col_name: str):
        series = pd.to_datetime(series)

        stats = {
            "dtype": series.dtype,
            "count": series.count(),
            "null_count": series.isna().sum(),
            "start_date": series.min(),
            "end_date": series.max(),
            "n_unique_dates": series.nunique(),
            "duplicate_dates": series.duplicated().sum()
        }

        # Missing dates detection (only if frequency can be inferred)
        try:
            full_range = pd.date_range(series.min(), series.max(), freq=pd.infer_freq(series.dropna()))
            stats["missing_dates"] = len(full_range.difference(series.dropna()))
        except Exception:
            stats["missing_dates"] = "Cannot infer frequency"

        if visualize:
            series.value_counts().sort_index().plot(
                figsize=(10, 4), title=f"{col_name} - Date Distribution"
            )
            plt.ylabel("Count")
            plt.show()

        return stats

    num_results = {}
    cat_results = {}
    date_results = {}
    unsupported_results = {}

    for col in df.columns:
        series = df[col]

        if (
            col in NUMERICAL
            or col in target
        ):
            num_results[col] = evaluate_numerical_feature(series, col)

        elif col in timestamp:
            date_results[col] = evaluate_datetime_feature(series, col)

        elif (
            col in BOOLEAN
            or col in CATEGORICAL
            or col is id
        ):
            cat_results[col] = evaluate_categorical_feature(series, col)

        else:
            unsupported_results[col] = {
                "dtype": series.dtype,
                "null_count": series.isna().sum(),
                "note": "Unsupported data type"
            }

    num_results = pd.DataFrame(num_results).T
    cat_results = pd.DataFrame(cat_results).T
    date_results = pd.DataFrame(date_results).T
    unsupported_results = pd.DataFrame(unsupported_results).T
    results = {
        'num': num_results,
        'cat': cat_results,
        'date': date_results,
        'unsupported': unsupported_results
    }
    
    return results


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations

def bivariate_analysis_mixed(
    df: pd.DataFrame,
    # features: list,
    sample_size: int = 10000
):
    """
    Draw appropriate bivariate plots for numerical, categorical, and boolean features.
    """

    features = [col for col in df.columns if col in FEATURES]
    plot_df = df.copy()

    if sample_size is not None and len(plot_df) > sample_size:
        plot_df = plot_df.sample(sample_size, random_state=42)

    for f1, f2 in combinations(features, 2):
        s1, s2 = plot_df[f1], plot_df[f2]
        
        # Identify feature types
        is_num1 = f1 in NUMERICAL
        is_num2 = f2 in NUMERICAL

        plt.figure(figsize=(6, 4))

        # Numerical vs Numerical
        if is_num1 and is_num2:
            sns.scatterplot(data=plot_df, x=f1, y=f2, alpha=0.5)

        # Numerical vs Categorical
        elif is_num1 and ~(is_num2):
            sns.boxplot(data=plot_df, x=f2, y=f1)

        elif ~(is_num1) and is_num2:
            sns.boxplot(data=plot_df, x=f1, y=f2)

        # Categorical vs Categorical
        elif ~(is_num1) and ~(is_num2):
            ct = pd.crosstab(plot_df[f1], plot_df[f2])
            sns.heatmap(ct, annot=True, fmt="d", cmap="Blues")

        plt.title(f"{f1} vs {f2}")
        plt.tight_layout()
        plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

def plot_time_and_seasonality(
    df: pd.DataFrame,
    value_col: str,
    date_col: str = "date",
    stat_type: str = 'mean',
    hue: str = None,
    timeplt_line: str = '-',
    freq: str = "D",
    decompose: bool = True
):
    """
    Draw multiple time-series plots for a numerical feature:
    - Time plot (supports multiple lines via hue)
    - Seasonal plots (year, month, week, day, year_month)
    - Optional seasonal decomposition (only when hue=None)
    """
    def _place_legend_below(ax, title=None, ncol=6):
        ax.legend(
            title=title,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.25),
            ncol=ncol,
            frameon=False
        )

    cols = [date_col, value_col]
    if hue:
        cols.append(hue)

    data = df[cols].copy()
    data[date_col] = pd.to_datetime(data[date_col])
    data = data.sort_values(date_col)

    # -----------------------------
    # 1. Time plot
    # -----------------------------
    # plt.figure(figsize=(12, 4))

    if hue:
        ax = plt.gca()
        
        for key, grp in data.groupby(hue):
            ax.plot(grp[date_col], grp[value_col], label=str(key), alpha=0.8)
        
        _place_legend_below(ax, title=hue, ncol=8)
        plt.subplots_adjust(bottom=0.3)

    else:
        plt.plot(data[date_col], data[value_col], timeplt_line)

    plt.title(f"Time Plot: {value_col}")
    plt.xlabel("Date")
    plt.ylabel(value_col)
    plt.tight_layout()
    plt.show()

    # Set datetime index for seasonal analysis
    data = data.set_index(date_col)

    # Create calendar features
    data["year"] = data.index.year
    data["month"] = data.index.month
    data["week"] = data.index.isocalendar().week.astype(int)
    data["day"] = data.index.day
    data["year_month"] = data.index.to_period("M")
    data["dayofweek"] = data.index.dayofweek + 1  # Monday=1, Sunday=7

    # -----------------------------
    # 2. Seasonal plots
    # -----------------------------
    seasonal_features = ["year", "month", "week", "day", "dayofweek", "year_month"]

    for feature in seasonal_features:
        plt.figure(figsize=(10, 4))

        if hue:
            if  stat_type == 'mean':
                grouped = (
                    data.groupby([feature, hue])[value_col]
                    .mean()
                    .unstack(hue)
                )
            else: 
                grouped = (
                    data.groupby([feature, hue])[value_col]
                    .sum()
                    .unstack(hue)
                )

            ax = grouped.plot()
            _place_legend_below(ax, title=hue, ncol=6)
            
            plt.title(f"Seasonal Plot ({feature}): {value_col}")
            plt.xlabel(feature)
            plt.ylabel(f"T {value_col}")
            plt.subplots_adjust(bottom=0.3)
            plt.show()

        else:
            if  stat_type == 'mean':
                grouped = data.groupby(feature)[value_col].mean()
            else:
                grouped = data.groupby(feature)[value_col].sum()
                
            grouped.plot()

            plt.title(f"Seasonal Plot ({feature}): {value_col}")
            plt.xlabel(feature)
            plt.ylabel(f"Total {value_col}")
            plt.tight_layout()
            plt.show()

    # -----------------------------
    # 3. Seasonal decomposition
    # -----------------------------
    if decompose and hue is None:
        try:
            ts = data[value_col].asfreq(freq).ffill()

            decomposition = seasonal_decompose(
                ts,
                model="additive",
                period=_infer_period(freq)
            )

            decomposition.plot()
            plt.suptitle(f"Seasonal Decomposition: {value_col}", y=1.02)
            plt.show()

        except Exception as e:
            print("Seasonal decomposition failed:", e)

    elif decompose and hue is not None:
        print("⚠️ Seasonal decomposition skipped (not valid for multiple series).")


def _infer_period(freq: str):
    """
    Infer seasonal period based on frequency.
    """
    if freq == "D":
        return 7      # weekly seasonality
    elif freq == "W":
        return 52
    elif freq == "M":
        return 12
    else:
        return None


# `stores`

In [ ]:
nbr_order = list(range(1, 55))
stores['store_nbr'] = stores['store_nbr'].cat.set_categories(nbr_order, ordered=True)

type_order = ['A', 'B', 'C', 'D', 'E']
stores['store_type'] = stores['store_type'].cat.set_categories(type_order, ordered=True)

cluster_order = list(range(1, 18))
stores['cluster'] = stores['cluster'].cat.set_categories(cluster_order, ordered=True)

In [ ]:
results = univariate_analysis(stores, visualize=False)
for name, result in results.items():
    print(f'# {name} #')
    display(result)

Both train and test datasets have the same 54 stores, no stores are added and removed.

In [ ]:
set(test['store_nbr'].unique()) ^ set(train['store_nbr'].unique())

# `Holiday`

All holidays happened in Ecuador.

In [ ]:
results = univariate_analysis(holidays, visualize=True)
for name, result in results.items():
    print(f'# {name} #')
    display(result)

In [ ]:
bivariate_analysis_mixed(holidays, None)

## Understand the locale distribution of Ecuador holidays

In [ ]:
local_h = holidays[holidays['locale']=='Local']
regional_h = holidays[holidays['locale']=='Regional']
national_h = holidays[holidays['locale']=='National']

In [ ]:
national_h['locale_name'].value_counts()

### Check geographical mapping between `holidays` and `stores`

Every local and regional holiday in the dataset is relevant, but not every city has a local holiday.

In [ ]:
all_city = stores['city'].unique().tolist()
all_state = stores['state'].unique().tolist() 

In [ ]:
all_local = local_h['locale_name'].unique().tolist()
set(all_city) ^ set(all_local) 

There are zero local holidays in `stores` df for cities where you don't have stores.

In [ ]:
set(all_local) - set(all_city)

These three cities (`Babahoyo`, `Daule`, `Playas`) have stores, but they never appear in the `holidays`.

In [ ]:
set(all_city) - set(all_local)

In [ ]:
all_regions = regional_h['locale_name'].unique().tolist()
set(all_state) ^ set(all_regions) 

There are zero local holidays in `stores` df for states where you don't have stores.

In [ ]:
set(all_regions) - set(all_state) 

These several states have stores, but they never appear in the `holidays`.

In [ ]:
set(all_state) - set(all_regions) 

### Geographic Nesting
- Multiple holidays can happened on the same day but at different locations (e.g., 2 different cities)
- A city can have its own "Local" holiday on the same day as a "National" holiday (e.g., a city foundation day falling on Christmas).

In [ ]:
overlapped_holidays = holidays[holidays['date'].duplicated()].sort_values(by='date')
overlapped_holidays

# `Oil`

There is no outlier for oil price

In [ ]:
results = univariate_analysis(oil, visualize=True)
for name, result in results.items():
    print(f'# {name} #')
    display(result)

In [ ]:
plot_time_and_seasonality(
    oil,
    'dcoilwtico'
)

### The missing values in `oil` are not "errors" or "random gaps." They are structural absences determined by the global financial calendar.

**Weekends**: Trading stops on Friday evening and resumes Sunday evening. Therefore, every Saturday and Sunday in your dataset will naturally have missing oil values.

In [ ]:
oil["day_of_week"]=oil["date"].dt.dayofweek+1
oil['day_of_week'].value_counts()

**Public Holidays**: Holidays (like Thanksgiving, Christmas, or New Year) result in market closures.

In [ ]:
day_no_price = oil[oil['dcoilwtico'].isna()]

holiday_dates = holidays['date'].unique().tolist()
day_no_price['is_holiday'] = day_no_price['date'].isin(holiday_dates) 

In [ ]:
display(day_no_price)

# txns

In [ ]:
results = univariate_analysis(txns, visualize=True)
for name, result in results.items():
    print(f'# {name} #')
    display(result)

### In Ecuador, Corporación Favorita stores are closed on Christmas Day.
- 2016-01-01: New Year only appears as a missing day in 2016.
- 2016-01-03: This is a random Sunday in January.

Sales records exists in in `train`for these dates, the missing records in `txns` is likely an anomaly or a data logging error.
However, `test` (Aug 16–31) does not include any of these missing-date anomalies. Therefore, we don't need to worry about predicting "zero" for a total shutdown during the test period.

In [ ]:
import pandas as pd

def find_missing_dates(df, date_col='date', freq='D'):
    """
    Identifies dates missing from a continuous sequence.
    
    Args:
        df (pd.DataFrame): Your dataset.
        date_col (str): The name of the column containing dates.
        freq (str): Frequency of the series ('D' for daily, 'B' for business days).
        
    Returns:
        pd.DatetimeIndex: The dates that are missing.
    """
    # 1. Ensure the column is in datetime format
    df[date_col] = pd.to_datetime(df[date_col])
    
    # 2. Define the "perfect" range from start to finish
    start_date = df[date_col].min()
    end_date = df[date_col].max()
    
    perfect_range = pd.date_range(start=start_date, end=end_date, freq=freq)
    
    # 3. Find the difference between the perfect range and your actual data
    missing_dates = perfect_range.difference(df[date_col])
    
    print(f"Analysis complete for {date_col}.")
    print(f'Missing dates: {missing_dates}')
    print(f"Total missing days: {len(missing_dates)}")
    
    return missing_dates

find_missing_dates(txns)

In [ ]:
newyear = train[train['date']==pd.to_datetime('2016-01-01')]
newyear[newyear['sales']!=0]

In [ ]:
newyear = train[train['date']==pd.to_datetime('2016-01-03')]
newyear[newyear['sales']!=0]

In [ ]:
for year in range(2015, 2019, 1):
    print(f'Christmas in year {year}')
    newyear = train[train['date']==pd.to_datetime(f'{year}-12-25')]
    display(newyear[newyear['sales']!=0])

- **Dual-Perspective Analysis**: The `plot_time_and_seasonality` function was executed twice—once using the mean transaction count to observe average store performance, and once using the sum to track the total volume of shoppers across the entire retail network.
- **Correlation with Oil Prices**: There is a noticeable decline in transaction counts starting in 2014. This aligns directly with the significant drop in oil prices beginning that same year, highlighting the sensitivity of Ecuadorian retail traffic to the nation's primary economic driver.
- **Annual Year-End Surge**: Transaction volume rises significantly every December. This "year-end effect" captures the massive seasonal spike in shopping activity during the holiday season.
- **Weekly Cycle Peaks**: Transactions are consistently at their highest during the weekends. This confirms that consumer behavior in Ecuador is heavily skewed toward Saturday and Sunday shopping trips.
- **Active Store Distortion (Sum vs. Mean)**:
  - The total sum of transactions remains relatively stable over time.
  - However, the mean (average per store) shows a noticeable drop starting in the second half of 2015.
  - Interpretation: This discrepancy is caused by new stores becoming active and entering the dataset in late 2015. Adding these new stores increases the denominator when calculating averages, revealing that store activation dates are staggered rather than simultaneous.

In [ ]:
plot_time_and_seasonality(
    txns,
    'transactions',
    timeplt_line = 'o'
)

In [ ]:
plot_time_and_seasonality(
    txns,
    'transactions',
    timeplt_line = 'o',
    stat_type = 'sum'
)

In [ ]:
# Check for duplicates across the unique combination of 'date' and 'store_nbr'
duplicates = txns.duplicated(subset=['date', 'store_nbr'], keep=False)

if duplicates.any():
    print(f"⚠️ Found {duplicates.sum()} duplicate rows!")
    print(txns[duplicates].sort_values(['date', 'store_nbr']).head())
else:
    print("✅ Success: Each store appears only once per day.")

- **Consistent Individual Trends**: When decomposing the data with `hue='store_nbr'`, the individual store trends largely mirror the global patterns, confirming that the weekend peaks and year-end surges are universal behaviors across the entire fleet.
- **Discrepancy between the mean and sum seasonal plots**:
  - [Sum] The Drop in 2017 is not a real sales crash. Since the training data ends on August 15, 2017, you only have about 7 months of data for that year. Naturally, the sum of transactions for 7 months will be significantly lower than the sum for a full 12-month year.
  - [Mean] The mean calculation automatically accounts for the fact that 2017 only has 7 months of data. It divides the total transactions by the number of days recorded. This is why the lines appear horizontal and stable—the "daily rhythm" of a store usually stays consistent year-over-year, even if the total annual volume changes.
  - [Sum] The "Zero" Starts: Several lines (like Store 52) starting at 0 in 2013 and only "rising" in 2015 or 2017. When you sum these, the total volume is heavily influenced by how many months of data are available for that specific calendar year.
  - [Mean] Opening Trends: Notice the lines that "start" in 2014 or 2015 in the mean plot. They often start at a specific value rather than 0 because the mean is only calculated for the days the store was actually active.
- **Staggered Market Entry**: The plots clearly visualize that different store_nbr identifiers begin their time series at different points on the x-axis. This confirms that the dataset contains "staggered starts" for new branch openings.
  - Example: Some stores show a full history from 2013, while others (like Store 52) only appear in the final years of the dataset.

In [ ]:
plot_time_and_seasonality(
    txns,
    'transactions',
    hue='store_nbr'
)

In [ ]:
plot_time_and_seasonality(
    txns,
    'transactions',
    hue='store_nbr',
    stat_type = 'sum'
)

### Cross-references between `train` (sales) and `txns` (foot traffic) 
- To pinpoint exactly when each store became active to handle the staggered start problem.
- All dates between `train` and `txns` match, it gives 100% confidence to use that specific date as the "start point" for that store. I safely discard all data rows before this date, removing the "pre-opening zeros" that would otherwise confuse  model.

In [ ]:
# 2. Find earliest date where sales > 0 for each store
# We filter for sales > 0 first to exclude the "pre-opening" zeros
first_sales = train[train['sales'] > 0].groupby('store_nbr')['date'].min().reset_index()
first_sales.columns = ['store_nbr', 'first_sales_date']

# 3. Find earliest date where transactions > 0 for each store
first_txns = txns[txns['transactions'] > 0].groupby('store_nbr')['date'].min().reset_index()
first_txns.columns = ['store_nbr', 'first_txn_date']

# 4. Create verification dataframe
# Start with a unique list of all stores
verification_df = pd.DataFrame({'store_nbr': sorted(train['store_nbr'].unique())})

# Merge the start dates
verification_df = verification_df.merge(first_sales, on='store_nbr', how='left')
verification_df = verification_df.merge(first_txns, on='store_nbr', how='left')

# Compare the date values
verification_df['dates_match'] = verification_df['first_sales_date'] == verification_df['first_txn_date']
verification_df['days_diff'] = (verification_df['first_sales_date'] - verification_df['first_txn_date']).dt.days

# Display the results
print("Store Start Date Verification:")
print(verification_df.sort_values('first_sales_date', ascending=False).head(10))

# Summary of discrepancies
mismatches = verification_df[verification_df['dates_match'] == False]
if not mismatches.empty:
    print(f"\nFound {len(mismatches)} stores where start dates do not match.")
    print(mismatches)
else:
    print("\nSuccess: All store start dates match across both datasets.")

# Optional: Save to CSV for external review
# verification_df.to_csv('store_start_verification.csv', index=False)

### Not all stores launch all product families on the very first day a store opens.
- Certain categories may have been added to an existing store's inventory years after the store's initial opening.
-  A product family might show 0 sales for months not because of a lack of demand, but because that specific category was not yet carried by that store.
-  Actionable: For each specific product line in each store, I remove the rows that occur before its "birth date." This prevents the model from trying to learn patterns from "structural zeros" (when the product didn't exist) versus "behavioral zeros" (when the product was out of stock or nobody bought it).

In [ ]:
import pandas as pd

# --------------------------------------------------
# 1. Store–Family activation dataframe
# --------------------------------------------------
store_family_start = (
    train[train["sales"] > 0]
    .groupby(["store_nbr", "family"])
    .agg(first_active_date=("date", "min"))
    .reset_index()
)

store_family_start.head()


In [ ]:
# --------------------------------------------------
# 2. Store-level consistency check
# --------------------------------------------------
store_start_consistency = (
    store_family_start
    .groupby("store_nbr")
    .agg(
        n_families=("family", "nunique"),
        n_unique_start_dates=("first_active_date", "nunique"),
        earliest_start_date=("first_active_date", "min"),
        latest_start_date=("first_active_date", "max"),
    )
    .reset_index()
)

# Boolean indicator
store_start_consistency["all_families_same_start"] = (
    store_start_consistency["n_unique_start_dates"] == 1
)

store_start_consistency.head()


In [ ]:
store_start_consistency['all_families_same_start'].value_counts()